============================================

**PROJECT:** Honey Yield Predictive Model

**TEAM:** The Beehive Team
- Stephanie Nord
- David Jorgensen
- Joshua Amaya

**COURSE:** DSC 630 Predictive Analytics

============================================

# Honey Yield Predictive Model

## Project Overview

This project will explore whether hive, weather, and regional agricultural data can be used to predict honey yield or yield-related outcomes. The analysis will include data loading, auditing, preprocessing, exploratory data analysis, feature engineering, predictive modeling, and final interpretation.

## 1. Environment Setup

Import the libraries needed for data handling, visualization, preprocessing, and modeling.

In [26]:
# Data handling
import os
import duckdb
import pandas as pd
import numpy as np
from dotenv import load_dotenv

# Visualization
import matplotlib.pyplot as plt
import seaborn as sns

# Modeling
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

# Display settings
pd.set_option("display.max_columns", None)
pd.set_option("display.width", 120)

# Optional style settings
sns.set_theme(style="whitegrid")

## 2. Data Loading

Load the confirmed project datasets. At this stage, the expected sources include hive-level data, weather-related data, and USDA/NASS honey production data.

> **TEAM NOTE — QUACK SERVER CONNECTION:**
>
> Data loads from a shared DuckDB instance via the Quack extension. Each team member needs a `.env` file in the repo root with:
>
> ```
> TAILNET_HOST=duckdb.your-tailnet.ts.net:9494
> QUACK_TOKEN=your_token_here
> ```
>
> The `.env` file is gitignored. Do not commit credentials.

In [27]:
load_dotenv()

quack_host = os.getenv("TAILNET_HOST")
quack_token = os.getenv("QUACK_TOKEN")

con = duckdb.connect('honey.duckdb')
con.execute("FORCE INSTALL quack")
con.execute("INSTALL httpfs")
con.execute("LOAD quack")
con.execute("LOAD httpfs")
con.execute("SET s3_region = 'us-east-1'")

try:
    con.execute("DETACH bees")
except Exception:
    pass

con.execute(f"""
    ATTACH 'quack:{quack_host}' AS bees (
        TOKEN '{quack_token}',
        DISABLE_SSL true
    )
""")

# List available tables in the remote 'bees' database
tables = con.execute("FROM bees.query('SHOW TABLES')").fetchdf()

print(tables)


                           name
0               HOBOS_flow_2017
1          HOBOS_flow_schwartau
2           HOBOS_flow_wurzburg
3           HOBOS_humidity_2017
4      HOBOS_humidity_schwartau
5       HOBOS_humidity_wurzburg
6        HOBOS_temperature_2017
7   HOBOS_temperature_schwartau
8    HOBOS_temperature_wurzburg
9             HOBOS_weight_2017
10       HOBOS_weight_schwartau
11        HOBOS_weight_wurzburg
12                    USDA_NASS
13      bob_inspection_all_info
14    bob_inspection_categories
15     bob_inspection_locations
16              bob_inspections
17         bob_sensor_processed
18      usda_tuscon_annotations
19           usda_tuscon_images


In [29]:
hobos_flow_2017_df = con.execute("FROM bees.query('SELECT * FROM HOBOS_flow_2017')").df()
# usda_df = con.execute("FROM bees.query('SELECT * FROM usda_tucson_data')").df()
# nass_df = con.execute("FROM bees.query('SELECT * FROM nass_honey_data')").df()

# Query NOAA GHCN-Daily, published as Parquet on AWS Open Data
# Example weather data query - Retrieve July 31 2007 weather data for Tucson, AZ (USW00023160)
weather_df = con.execute("SELECT ID, strptime(DATE, '%Y%m%d')::DATE AS day, DATA_VALUE / 10.0 AS tmax_c FROM read_parquet('s3://noaa-ghcn-pds/parquet/by_station/STATION=USW00023160/ELEMENT=TMAX/*.parquet') WHERE DATE = '20170731' ORDER BY day;").df()

## 3. Initial Data Audit

Review the structure, size, data types, missing values, duplicate records, and general quality of each dataset.

In [3]:
def audit_dataframe(df, name="DataFrame"):
    """
    Print a basic audit summary for a DataFrame.
    """
    print(f"--- {name} Audit ---")
    print(f"Shape: {df.shape}")
    print("\nData Types:")
    print(df.dtypes)
    print("\nMissing Values:")
    print(df.isna().sum())
    print("\nDuplicate Rows:")
    print(df.duplicated().sum())
    print("\nPreview:")
    display(df.head())

In [ ]:
# TODO: Run once data is loaded
# audit_dataframe(hobos_df, "HOBOS Hive Data")
# audit_dataframe(usda_df, "USDA Tucson Data")
# audit_dataframe(nass_df, "NASS Honey Data")

## 4. Data Cleaning and Preprocessing

This section will prepare the datasets for analysis and modeling.

Planned tasks:
- Handle missing values
- Standardize column names
- Convert date/time fields
- Resample high-frequency hive data if needed
- Align datasets with different levels of granularity
- Merge datasets into a modeling-ready dataset

In [4]:
def clean_column_names(df):
    """
    Standardize column names for easier analysis.
    """
    df = df.copy()
    df.columns = (
        df.columns
        .str.strip()
        .str.lower()
        .str.replace(" ", "_")
        .str.replace("-", "_")
    )
    return df

In [ ]:
# TODO: Apply after loading data
# hobos_df = clean_column_names(hobos_df)
# usda_df = clean_column_names(usda_df)
# nass_df = clean_column_names(nass_df)

## 5. Feature Engineering

Create variables that may improve model performance, such as time-based features, weather summaries, hive weight changes, or regional production indicators.

TODO:
- [ ] Create date/time features
- [ ] Calculate hive weight change
- [ ] Aggregate weather variables
- [ ] Add regional/state-level honey production features

## 6. Exploratory Data Analysis

Use visualizations and summary statistics to understand patterns in the data before modeling.

Possible visualizations:
- Hive weight over time
- Temperature and humidity trends
- Relationship between hive weight and weather variables
- Annual honey production trends
- Missing data patterns

In [ ]:
# Example placeholder visualization
# plt.figure(figsize=(10, 6))
# sns.scatterplot(data=model_df, x="temperature", y="yield")
# plt.title("Temperature vs. Honey Yield")
# plt.xlabel("Temperature")
# plt.ylabel("Honey Yield")
# plt.show()

## 7. Modeling

Build predictive models using the cleaned and merged dataset.

Possible models:
- Linear Regression
- Decision Tree Regressor
- Random Forest Regressor
- Gradient Boosting Regressor

The final model choice will depend on the target variable, dataset size, and feature structure.

In [ ]:
# TODO:
# Define target variable
# y = model_df["target_variable"]

# Define features
# X = model_df.drop(columns=["target_variable"])

# Split data
# X_train, X_test, y_train, y_test = train_test_split(
#     X, y, test_size=0.2, random_state=42
# )

## 8. Model Evaluation

Evaluate model performance using regression metrics.

In [5]:
def evaluate_regression_model(y_true, y_pred, model_name="Model"):
    """
    Evaluate a regression model using common performance metrics.
    """
    mae = mean_absolute_error(y_true, y_pred)
    rmse = np.sqrt(mean_squared_error(y_true, y_pred))
    r2 = r2_score(y_true, y_pred)

    print(f"--- {model_name} Evaluation ---")
    print(f"MAE:  {mae:.3f}")
    print(f"RMSE: {rmse:.3f}")
    print(f"R²:   {r2:.3f}")

## 9. Results and Interpretation

Summarize the strongest predictors, model performance, limitations, and any important findings from the analysis.

## 10. Conclusions and Submission Preparation

Final project takeaways:

- What problem did the project address?
- What datasets were used?
- What preprocessing steps were required?
- Which model performed best?
- What insights were gained?
- What limitations should be noted?
- What future improvements could be made?